In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

In [6]:
RESULT_PATH = "../figures/SolarCells_1.html"
FILE_PATH = "../data/institution_network_evolution_SolarCells_2000_2025.csv"

In [7]:
# Load the file
df = pd.read_csv(FILE_PATH)

In [8]:
# Compute a center for the map
all_lats = pd.concat([df["source_lat"], df["target_lat"]], ignore_index=True)
all_lngs = pd.concat([df["source_lng"], df["target_lng"]], ignore_index=True)
center = [all_lats.mean(), all_lngs.mean()]

In [9]:
# Create map
m = folium.Map(location=center, zoom_start=2, tiles="OpenStreetMap")

In [10]:
# Build a table of all unique nodes
points = pd.concat([
    df[["source_id", "source_lat", "source_lng"]].rename(
        columns={"source_id": "node_id", "source_lat": "lat", "source_lng": "lng"}
    ),
    df[["target_id", "target_lat", "target_lng"]].rename(
        columns={"target_id": "node_id", "target_lat": "lat", "target_lng": "lng"}
    )
], ignore_index=True).drop_duplicates()

In [11]:
# Add markers
cluster = MarkerCluster(name="Institutions").add_to(m)

for _, row in points.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=3,
        popup=str(row["node_id"]),
        fill=True,
        fill_opacity=0.8,
    ).add_to(cluster)

In [12]:
# Add edges
for _, row in df.iterrows():
    folium.PolyLine(
        locations=[
            [row["source_lat"], row["source_lng"]],
            [row["target_lat"], row["target_lng"]],
        ],
        weight=max(1, float(row["weight"])),
        opacity=0.35,
        popup=f"{row['source_id']} → {row['target_id']} ({int(row['publication_year'])})",
    ).add_to(m)

folium.LayerControl().add_to(m)

In [13]:
# Save to HTML
m.save(RESULT_PATH)